# 05 — A demanding case: breast cancer

*Notebook 5 of 5 — Decision Trees*

This is where the chapter comes together. We put a single decision tree to work on a real diagnostic
problem — 569 patients, 30 measurements — and ask the two questions that matter for a tree: how
**accurate** is it, and how **interpretable**? The honest answers (a readable rule set, but less
accurate and unstable) are exactly what sets up the rest of the course.

**Prerequisites**

- Notebooks 01–04 — the whole method: impurity, growth, pruning, the estimator and its variance.
- Chapter 01 NB 5 (k-NN felt the curse on this same dataset) and Chapter 03 NB 6 (logistic regression
  read calibrated probabilities from it).
- Module 00 — the confusion matrix and recall (NB 07), cross-validation (NB 10), the train/test split
  (NB 04).

**What you'll be able to do**

- Run an honest decision-tree workflow end to end on real data.
- Read a fitted tree as a clinical rule set, and weigh interpretability against accuracy.
- See a single tree's variance and read feature importance honestly (Gini vs permutation).
- Explain why the next chapters average and boost trees.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

from ml_course import datasets, viz
from ml_course.colors import COLORS

np.random.seed(0)
viz.use_course_style()
cv = StratifiedKFold(5, shuffle=True, random_state=0)

df = datasets.load_breast_cancer()
X = df.drop(columns="target")
y = (df["target"] == 0).astype(int).to_numpy()  # malignant = 1 = the costly miss

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)
print(f"{len(y)} patients, {X.shape[1]} features; malignant {y.sum()} ({y.mean():.1%})")
print(f"train {len(ytr)}, test {len(yte)} (malignant in the test set: {yte.sum()})")

## Where we are, and the stakes

569 patients, 30 measurements per tumour, malignant 212 / benign 357. This is the dataset k-nearest
neighbours felt the curse of dimensionality on (Chapter 01) and logistic regression read calibrated
probabilities from (Chapter 03). Now we read it as a **tree** — a set of yes/no questions.

Two questions guide us: how **accurate** is the tree, and how **interpretable**? Because a missed
cancer costs far more than a false alarm, we watch **recall on the malignant class**, not accuracy
alone. And we do no standardizing — a tree does not need it (Notebook 04).

In [ ]:
fig = viz.plot_class_balance(pd.Series(y).map({0: "benign", 1: "malignant"}))
plt.show()

df[["mean concave points", "worst area", "worst concave points"]].describe().round(2)

### Read the figure

About 37% of the patients are malignant — not balanced. A model that always guessed "benign" would
score 63% accuracy while catching zero cancers, so accuracy alone would flatter it. We will judge the
tree on how many malignant tumours it actually catches (recall), not on accuracy by itself.

## The honest workflow

We compare a tree against the Chapter-03 logistic-regression baseline by **cross-validating on the
training set**, tune the tree's dials on the training set, and read the **sealed test set once** at the
end. Same discipline as module 00 — no peeking.

In [ ]:
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=0), Xtr, ytr, cv=cv).mean()
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
logreg_cv = cross_val_score(logreg, Xtr, ytr, cv=cv).mean()
print(f"CV-on-train:  decision tree {tree_cv:.3f}   |   logistic regression {logreg_cv:.3f}")

grid = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "criterion": ["gini", "entropy"],
    "ccp_alpha": [0.0, 0.005, 0.01],
}
tuned = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=cv).fit(Xtr, ytr)
logreg.fit(Xtr, ytr)
tree_test, logreg_test = tuned.score(Xte, yte), logreg.score(Xte, yte)
print(f"tuned tree best params: {tuned.best_params_}")
print(f"sealed test:  tree {tree_test:.3f}   |   logistic regression {logreg_test:.3f}")

### Read the result

The tree trails the linear model by four to five points — in cross-validation (0.94 vs 0.98) and on the
sealed test (0.91 vs 0.95). For raw accuracy on this data, logistic regression wins. But accuracy is
not the whole story: the next cells ask what the tree can *tell* us that the linear model cannot.

## Interpretability: a tree is a rule set

A logistic regression hands you 30 weights; k-nearest neighbours hands you distances to neighbours.
A shallow tree hands you something a person can read and challenge: a short flowchart of yes/no
questions. Let us grow a depth-3 tree and read its rules.

In [ ]:
depth3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(f"depth-3 tree: {depth3.get_n_leaves()} leaves, test accuracy {depth3.score(Xte, yte):.3f}\n")
print(export_text(depth3, feature_names=list(X.columns)))

fig, ax = plt.subplots(figsize=(12, 7))
plot_tree(
    depth3,
    feature_names=list(X.columns),
    class_names=["benign", "malignant"],
    filled=False,
    rounded=True,
    fontsize=9,
    ax=ax,
)
plt.show()

### Read the figure

The rules read like clinical criteria. The first question is about `mean concave points` (how many
sharp indentations the tumour boundary has); below it come tumour **size** (`worst area`) and **shape**
(`area error`, `worst fractal dimension`, `worst concave points`). Larger, more irregular tumours fall
to the malignant leaves. A clinician could read this tree, agree or argue with each threshold, and audit a
prediction — interpretability k-NN, naive Bayes, and logistic regression could not offer in this form.

## Interpretability versus accuracy

Line the methods up on the same sealed test set — and add one preview of what comes next.

In [ ]:
# k-NN: pick k by cross-validation on train, then score the sealed test.
ks = [3, 5, 7, 9, 11]
best_k = max(
    ks,
    key=lambda k: cross_val_score(
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), Xtr, ytr, cv=cv
    ).mean(),
)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(Xtr, ytr)

# Hand-bag 25 bootstrap trees and take the majority vote (a random forest in miniature).
rng = np.random.default_rng(1)
Xa, Xb = Xtr.to_numpy(), Xte.to_numpy()
votes = []
for _ in range(25):
    idx = rng.integers(0, len(Xa), len(Xa))
    votes.append(DecisionTreeClassifier(random_state=0).fit(Xa[idx], ytr[idx]).predict(Xb))
# 25 voters: mean >= 0.5 is a strict majority (>= 13 of 25), with no ties.
bagged_test = ((np.mean(votes, axis=0) >= 0.5).astype(int) == yte).mean()

methods = [f"k-NN (k={best_k})", "logistic reg.", "single tree", "bagged 25 trees"]
scores = [knn.score(Xte, yte), logreg.score(Xte, yte), tuned.score(Xte, yte), bagged_test]
bar_colors = [COLORS["muted"], COLORS["muted"], COLORS["highlight"], COLORS["correct"]]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(methods, scores, color=bar_colors, edgecolor=COLORS["text"], linewidth=0.6)
for i, s in enumerate(scores):
    ax.text(i, s, f"{s:.3f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylim(0.85, 0.97)
ax.set_ylabel("test accuracy")
plt.show()

### Read the figure

Among the three single models, the tree (in pink) is the **most interpretable but the least accurate**
— that is the trade you accept for a readable rule set. The last bar is a preview of the fix: bag 25
trees (fit each on a resample, take the majority vote) and accuracy climbs from 0.906 to 0.930 —
closing about half the gap to the linear model. Averaging many trees is the ensemble idea the next
chapters are built on.

## Where a single tree fails: variance

Notebook 04 showed a single tree is high-variance on the moons. On real 30-feature data it is starker:
watch even the tree's **first question** as we resample the patients.

In [ ]:
rng = np.random.default_rng(0)
roots, boot_acc = [], []
for _ in range(25):
    idx = rng.integers(0, len(Xa), len(Xa))
    tree = DecisionTreeClassifier(random_state=0).fit(Xa[idx], ytr[idx])
    roots.append(X.columns[tree.tree_.feature[0]])
    boot_acc.append((tree.predict(Xb) == yte).mean())

root_counts = pd.Series(roots).value_counts()
print(f"bootstrap test accuracy: mean {np.mean(boot_acc):.3f}, std {np.std(boot_acc):.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ordered = root_counts.sort_values()
ax.barh(
    ordered.index,
    ordered.to_numpy(),
    color=COLORS["model"],
    edgecolor=COLORS["text"],
    linewidth=0.6,
)
ax.set_xlabel("times chosen as the root split (out of 25 resamples)")
plt.show()

### Read the figure

The very first split is not stable. Across 25 resamples of the same patients, the tree opens on
`mean concave points` most often — but on a fair number of resamples it opens on `worst perimeter`,
`worst concave points`, or `worst radius` instead. If even the root question changes with the sample,
the whole tree below it does too; the held-out accuracy wobbles by a couple of points (std ≈ 0.02).
That instability is the single tree's defining weakness — and exactly what averaging trees removes.

## Reading importance honestly: Gini versus permutation

Notebook 04 warned that the built-in (Gini) feature importance is biased toward features with many
distinct values. Here is that warning made concrete. We compare it against **permutation importance**,
which shuffles one feature at a time and measures how much test accuracy drops — a direct measure of
how much the model actually relies on each feature.

In [ ]:
full_tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
gini_imp = pd.Series(full_tree.feature_importances_, index=X.columns)
perm = permutation_importance(tuned.best_estimator_, Xte, yte, n_repeats=10, random_state=0)
perm_imp = pd.Series(perm.importances_mean, index=X.columns)

print("Gini top 3       :", list(gini_imp.nlargest(3).index))
print("Permutation top 3:", list(perm_imp.nlargest(3).index))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
g = gini_imp.nlargest(6).sort_values()
ax1.barh(g.index, g.to_numpy(), color=COLORS["class_a"], edgecolor=COLORS["text"], linewidth=0.6)
ax1.set_title("Gini importance (full tree)")
ax1.set_xlabel("share of impurity decrease")
p = perm_imp.nlargest(6).sort_values()
ax2.barh(p.index, p.to_numpy(), color=COLORS["class_b"], edgecolor=COLORS["text"], linewidth=0.6)
ax2.set_title("permutation importance (test)")
ax2.set_xlabel("accuracy drop when shuffled")
fig.tight_layout()
plt.show()

### Read the figure

The two measures **disagree**. Gini importance crowns `mean concave points` (about 0.74 of the total)
— the feature the tree happened to split on first, which then gets credited for nearly everything
below it. Permutation importance, measuring actual reliance on held-out data, puts `worst area` and
`worst concave points` first and does not rank `mean concave points` at the top at all. When features
are correlated (and these size/shape measurements are), Gini over-credits whichever one was chosen
first. Trust the permutation read, or at least cross-check the two before telling a story about "the
most important feature."

In [ ]:
cm = confusion_matrix(yte, tuned.predict(Xte))
fig = viz.plot_confusion_matrix(cm, class_names=["benign", "malignant"])
plt.show()

recall = cm[1, 1] / cm[1].sum()
missed, fp = cm[1, 0], cm[0, 1]
print(f"malignant recall {recall:.3f}: {missed}/{cm[1].sum()} cancers missed, {fp} false alarms")

### Read the figure

On the sealed test the tuned tree misses **4 of 64 cancers** (recall 0.94) and raises 12 false alarms.
The misses are borderline tumours where the rule thresholds are too coarse. At the same default
threshold, Chapter 03's logistic regression missed the *same* 4 cancers but raised far fewer false
alarms (4, not 12); and because its probabilities are calibrated, lowering its threshold could cut the
misses to 1 (at the price of more false alarms) — a dial a single hard-rule tree does not offer. What
the tree gives in return is the readable rule set. That is the interpretability-versus-accuracy trade,
made concrete on patients.

## Error analysis and honest limits

A single decision tree on this problem is **interpretable** (a clinician-readable rule set),
**scale-free**, and comfortable with missing values and many classes (Notebook 04). Its costs are
equally clear: it is **less accurate** than the linear model here, and **high-variance** — even its
first question changes with the sample.

- **When to use a single tree:** you need a model a person can read and audit, you have mixed or
  unscaled features, or you want a fast, honest baseline.
- **When not:** you need the last few points of accuracy, stable feature importances, or a calibrated
  probability — then **average** trees (random forests, Chapter 06) or **boost** them (Chapters 07–10).

The bagged-25 bar a few cells up was that fix in miniature: the variance you saw above is precisely
what an ensemble averages away.

## Your turn

1. **Trace the rules.** Take a patient with `mean concave points = 0.08`, `worst concave points = 0.15`,
   `worst area = 800`. Follow the printed depth-3 rules from the top and give the predicted class.
2. **Two importances.** List the Gini top-3 and the permutation top-3 and explain, in two sentences,
   why a feature can top one ranking and not the other.
3. **Build a forest by hand.** Bag K bootstrap trees with a majority vote, sweep `K = 1, 5, 25`, and
   plot test accuracy against K. You have built the core idea of a random forest — confirm the
   variance falls and the accuracy rises as K grows.

## What you built

- An honest decision-tree workflow on real diagnostic data: cross-validated model choice, a sealed
  test, and recall as the metric that matters under imbalance.
- The tree's **readable rule set** — and the measured **interpretability-versus-accuracy** trade
  against k-NN and logistic regression.
- A single tree's **high variance** (its root question flips across resamples) and the honest reading
  of feature importance (**Gini versus permutation**).
- The **ensemble bridge**: a hand-bagged set of 25 trees that recovers most of the accuracy gap.

**Vocabulary:** interpretability vs accuracy · readable rule set · variance / instability · Gini vs
permutation importance · bagging · ensemble.

## Chapter wrap — decision trees, end to end

From a single question (Notebook 01) to a grown, pruned, tuned, and *read* tree on real patients
(Notebook 05), you have built the decision tree by hand and met it as an estimator. It is the course's
first **non-linear**, rule-based method: it carves the feature space into axis-aligned boxes, needs no
scaling, handles many classes and missing values, and — uniquely so far — explains itself.

Its weakness is equally clear: a single tree is **unstable**, and a touch less accurate than a tuned
linear model. That weakness is the doorway to the rest of the course. The next module turns to
**margins** (Module 05 — Support Vector Machines); then the fix for the variance you saw here arrives
in **Module 06 — Random Forests**, which averages many trees, followed by the boosting family
(Chapters 07–10). The humble tree you built is the base learner for all of them.

## References

- Breiman, Friedman, Olshen & Stone (1984). *Classification and Regression Trees (CART).* Wadsworth.
- Breiman, L. (1996). *Bagging predictors.* Machine Learning 24, 123–140. DOI: 10.1007/BF00058655.
- Strobl, C., Boulesteix, A.-L., Zeileis, A. & Hothorn, T. (2007). *Bias in random forest variable
  importance measures.* BMC Bioinformatics 8:25. DOI: 10.1186/1471-2105-8-25.
- Street, W. N., Wolberg, W. H. & Mangasarian, O. L. (1993). *Nuclear feature extraction for breast
  tumor diagnosis.* Proc. SPIE 1905. DOI: 10.1117/12.148698.
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, §9.2.
  DOI: 10.1007/978-0-387-84858-7.
- James, Witten, Hastie & Tibshirani (2021). *An Introduction to Statistical Learning*, §8.1.
  DOI: 10.1007/978-1-0716-1418-1.

*Previous: 04 — The estimator & its parameters · Next: Module 05 — Support Vector Machines.*